In [2]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import librosa
from datasets import Dataset
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC, TrainingArguments, Trainer
import evaluate
from functools import partial

c:\Users\Madesh\Desktop\Data Science Project\Noisy_Speech_Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [4]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

CLEAN_MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "clean", "final_model")
NOISY_MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "noisy", "final_model")

BASE_PATH = "../data/clean/LibriSpeech/train-clean-100"
NOISE_PATH = "../data/musan/noise"

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

clean_model = Wav2Vec2ForCTC.from_pretrained(CLEAN_MODEL_PATH).to(device)
noisy_model = Wav2Vec2ForCTC.from_pretrained(NOISY_MODEL_PATH).to(device)

Loading weights: 100%|██████████| 213/213 [00:00<00:00, 472.00it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


In [6]:
data = []

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:
        if file.endswith(".trans.txt"):
            trans_path = os.path.join(root, file)
            with open(trans_path, "r") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split(" ", 1)
                file_id = parts[0]
                text = parts[1].lower()
                audio_path = os.path.join(root, file_id + ".flac")

                if os.path.exists(audio_path):
                    data.append({
                        "audio_path": audio_path,
                        "text": text
                    })

df = pd.DataFrame(data)
print("Total samples:", len(df))

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z ']", "", text)
    return text

df["clean_text"] = df["text"].apply(normalize_text)
df_sample = df.sample(1000, random_state=42).reset_index(drop=True)

Total samples: 28539


In [7]:
TARGET_SR = 16000

def load_audio(path):
    y, sr = librosa.load(path, sr=None)
    if sr != TARGET_SR:
        y = librosa.resample(y, orig_sr=sr, target_sr=TARGET_SR)
    return y.astype("float32")

In [8]:
noise_files = []
for root, dirs, files in os.walk(NOISE_PATH):
    for file in files:
        if file.endswith(".wav"):
            noise_files.append(os.path.join(root, file))

In [9]:
def add_noise(clean, noise, snr_db):

    if len(noise) < len(clean):
        repeat = int(np.ceil(len(clean) / len(noise)))
        noise = np.tile(noise, repeat)

    noise = noise[:len(clean)]

    clean_power = np.mean(clean ** 2)
    noise_power = np.mean(noise ** 2)
    snr = 10 ** (snr_db / 10)

    scale = np.sqrt(clean_power / (snr * noise_power))
    return clean + scale * noise

In [10]:
SNR_LEVELS = [None, 20, 10, 5, 0]
RATE_LEVELS = [0.8, 1.0, 1.2]
PITCH_LEVELS = [-2, 0, 2]

In [11]:
def apply_perturbations(audio, snr_db=None, rate=1.0, pitch_steps=0):

    # Rate
    if rate != 1.0:
        audio = librosa.effects.time_stretch(audio, rate)

    # Pitch
    if pitch_steps != 0:
        audio = librosa.effects.pitch_shift(audio, sr=16000, n_steps=pitch_steps)

    # Noise
    if snr_db is not None:
        noise_audio = load_audio(random.choice(noise_files))
        audio = add_noise(audio, noise_audio, snr_db)

    return audio

In [12]:
def apply_perturbations(audio, snr_db=None, rate=1.0, pitch_steps=0):

    # Rate
    if rate != 1.0:
        audio = librosa.effects.time_stretch(audio, rate=rate)

    # Pitch
    if pitch_steps != 0:
        audio = librosa.effects.pitch_shift(audio, sr=16000, n_steps=pitch_steps)

    # Noise
    if snr_db is not None:
        noise_audio = load_audio(random.choice(noise_files))
        audio = add_noise(audio, noise_audio, snr_db)

    return audio

In [13]:
def prepare_controlled_dataset(example, snr, rate, pitch):

    audio = load_audio(example["audio_path"])
    audio = apply_perturbations(audio, snr, rate, pitch)

    inputs = processor(audio, sampling_rate=16000)
    labels = processor(text=example["clean_text"].upper()).input_ids

    example["input_values"] = inputs.input_values[0]
    example["labels"] = labels

    return example

In [14]:
test_dataset = Dataset.from_pandas(df_sample)

condition_fn = partial(
    prepare_controlled_dataset,
    snr=10,
    rate=1.2,
    pitch=2
)

test_dataset = test_dataset.map(
    condition_fn,
    remove_columns=test_dataset.column_names
)

Map: 100%|██████████| 1000/1000 [02:34<00:00,  6.47 examples/s]


In [15]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    pred_str = processor.batch_decode(pred_ids, group_tokens=True)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

In [16]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:

        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"] != 1, -100
        )

        batch["labels"] = labels

        return batch

In [17]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [18]:
eval_args = TrainingArguments(
    output_dir="./temp_eval",
    per_device_eval_batch_size=4
)

clean_trainer = Trainer(
    model=clean_model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [19]:
results = clean_trainer.evaluate(test_dataset)

print("Composite Condition WER:", results["eval_wer"])
print("Composite Condition CER:", results["eval_cer"])

Composite Condition WER: 0.8258213923867419
Composite Condition CER: 0.5355073609165582


In [21]:
noisy_trainer = Trainer(
    model=noisy_model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

results_noisy = noisy_trainer.evaluate(test_dataset)

print("Noisy Model Composite WER:", results_noisy["eval_wer"])


Noisy Model Composite WER: 0.8112896222318715


In [22]:
conditions = {
    "clean": {"snr": None, "rate": 1.0, "pitch": 0},
    "noise_only": {"snr": 10, "rate": 1.0, "pitch": 0},
    "rate_only": {"snr": None, "rate": 1.2, "pitch": 0},
    "pitch_only": {"snr": None, "rate": 1.0, "pitch": 2},
}

In [23]:
clean_trainer = Trainer(
    model=clean_model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

noisy_trainer = Trainer(
    model=noisy_model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [24]:
results_table = []

for name, params in conditions.items():

    print("\n==============================")
    print("Running condition:", name)
    print(params)

    dataset = Dataset.from_pandas(df_sample)

    condition_fn = partial(
        prepare_controlled_dataset,
        snr=params["snr"],
        rate=params["rate"],
        pitch=params["pitch"]
    )

    dataset = dataset.map(
        condition_fn,
        remove_columns=dataset.column_names
    )

    clean_results = clean_trainer.evaluate(dataset)
    noisy_results = noisy_trainer.evaluate(dataset)

    results_table.append({
        "Condition": name,
        "Clean_Model_WER": clean_results["eval_wer"],
        "Noisy_Model_WER": noisy_results["eval_wer"]
    })


Running condition: clean
{'snr': None, 'rate': 1.0, 'pitch': 0}


Map: 100%|██████████| 1000/1000 [00:06<00:00, 144.81 examples/s]



Running condition: noise_only
{'snr': 10, 'rate': 1.0, 'pitch': 0}


Map: 100%|██████████| 1000/1000 [00:09<00:00, 103.57 examples/s]



Running condition: rate_only
{'snr': None, 'rate': 1.2, 'pitch': 0}


Map: 100%|██████████| 1000/1000 [00:54<00:00, 18.27 examples/s]



Running condition: pitch_only
{'snr': None, 'rate': 1.0, 'pitch': 2}


Map: 100%|██████████| 1000/1000 [01:12<00:00, 13.85 examples/s]


In [25]:
results_df = pd.DataFrame(results_table)
print(results_df)

    Condition  Clean_Model_WER  Noisy_Model_WER
0       clean         0.012013         0.015429
1  noise_only         0.066406         0.022232
2   rate_only         0.326154         0.335475
3  pitch_only         0.218729         0.248459
